# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

We will use **Logistic Regression**. Our problem is a 'which first?' ranking task (evaluated via Precision@K). Logistic regression naturally outputs readable probabilities (`predict_proba`) that we can use as a ranking score. It is simple, interpretable, and perfect for setting our first ML baseline without overfitting.

In [9]:
# Code cell not needed for this section, but left empty to satisfy the skeleton
print('Method chosen: Logistic Regression')

Method chosen: Logistic Regression


## 2. Split design

We will use a **Grouped Split** by `client_id` (using `GroupShuffleSplit`). This is honest because it ensures our model actually learns signals that apply to *new* clients, rather than memorizing the quirks and specific traffic patterns of the clients it was trained on.

In [10]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Load data (local paths first, then GitHub for Colab)
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = 'https://raw.githubusercontent.com/mohamed-6513/flyrank_ml/main/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# Filter to pages with meaningful traffic (same as baseline)
df = df[df['impressions_prev_30d'] > 100].copy()
print(f'Loaded {len(df)} pages.')

# Label: is declining
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Group Shuffle Split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"Train set: {len(df_train)} pages, {df_train['client_id'].nunique()} clients")
print(f"Test set: {len(df_test)} pages, {df_test['client_id'].nunique()} clients")

Loaded 17980 pages.
Train set: 14418 pages, 21 clients
Test set: 3562 pages, 9 clients


## 3. Train + compare vs my baseline

We will use a focused subset of features: the original 3 from the baseline (`content_age_days`, `competition`, `impressions_prev_30d`) plus `search_volume` and `sessions_prev_30d`. Both the model and the baseline will be evaluated on the same test split using Precision@K.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Features
features = ['content_age_days', 'competition', 'impressions_prev_30d', 'search_volume', 'sessions_prev_30d']
target = 'is_declining_label'

# Train Logistic Regression
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, class_weight='balanced'))
])
model.fit(df_train[features], df_train[target])

# Predict on test
df_test['model_prob'] = model.predict_proba(df_test[features])[:, 1]

# Recreate baseline score on test set
df_test['is_young'] = (df_test['content_age_days'] < 180).astype(int)
df_test['is_high_comp'] = (df_test['competition'] > 0.71).astype(int)
df_test['baseline_score'] = df_test[['is_young', 'is_high_comp']].max(axis=1) * df_test['impressions_prev_30d']

# Precision@K Evaluation
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df_test['is_declining_label'].mean()
print(f"Base rate (Test set): {base_rate:.4f}\n")

print(f"{'K':<5} | {'Baseline P@K':<15} | {'Model P@K':<15}")
print("-" * 40)
for k in [20, 50, 100, 500]:
    p_baseline = precision_at_k(df_test['baseline_score'], df_test['is_declining_label'], k)
    p_model = precision_at_k(df_test['model_prob'], df_test['is_declining_label'], k)
    print(f"{k:<5} | {p_baseline:.4f}          | {p_model:.4f}")

Base rate (Test set): 0.5665

K     | Baseline P@K    | Model P@K      
----------------------------------------
20    | 0.6500          | 0.8000
50    | 0.5800          | 0.7600
100   | 0.6000          | 0.6600
500   | 0.6720          | 0.6700


## 4. Errors and interpretation

We inspect the feature coefficients to see what drives the prediction, and we look at the top false positives where the model was wrong.

In [12]:
# 1. Feature Importance
coefs = model.named_steps['clf'].coef_[0]
importance = pd.DataFrame({'Feature': features, 'Coefficient': coefs})
importance = importance.sort_values(by='Coefficient', ascending=False)
print("Feature Coefficients (Positive = Drives 'Declining' prediction):")
display(importance)

# 2. Error Analysis: Top False Positives
df_test_sorted = df_test.sort_values(by='model_prob', ascending=False)
false_positives = df_test_sorted[df_test_sorted['is_declining_label'] == 0].head(3)

print("\nTop 3 False Positives (Model predicted high risk of decline, but page was stable/up):")
display(false_positives[['content_id', 'client_id', 'model_prob', 'trend_direction'] + features])

Feature Coefficients (Positive = Drives 'Declining' prediction):


,Feature,Coefficient
2,impressions_prev_30d,0.040743
1,competition,0.011191
3,search_volume,-0.006132
4,sessions_prev_30d,-0.416142
0,content_age_days,-0.430252



Top 3 False Positives (Model predicted high risk of decline, but page was stable/up):


,content_id,client_id,model_prob,trend_direction,content_age_days,competition,impressions_prev_30d,search_volume,sessions_prev_30d
6903,content_c84a0ab98e90,client_f369cb89fc,0.719909,stable,95,0.0,84773,0.0,19
22028,content_73c54f78c06a,client_f369cb89fc,0.695936,stable,97,0.0,97200,10.0,44
7180,content_f6ae0f36d70d,client_f369cb89fc,0.662289,stable,95,0.0,11128,0.0,1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.